# Stripmap PFA — Subaperture Image Formation for Extended Collections

**Objective**: Form a continuous strip SAR image from long stripmap CPHD collections using overlapping sub-aperture PFA and mosaicking.

## Overview

**Stripmap PFA** extends the Polar Format Algorithm to long stripmap collections by:
1. Partitioning the full aperture into **overlapping sub-apertures**
2. Forming each sub-aperture with PFA
3. **Mosaicking** the sub-images into a continuous strip with weighted blending

## Theory

### Sub-aperture Processing

A stripmap collection has phase history over a long track ($N$ pulses). Direct PFA over the full aperture would produce:
- Polar grid with excessive curvature (violates PFA assumptions)
- Non-uniform resolution across scene

**Solution**: Partition into blocks of $M$ pulses ($M \ll N$) with overlap $\alpha$:
$$\text{overlap fraction} = \alpha \in [0, 1)$$
$$\text{step size} = M(1 - \alpha)$$

### Mosaicking with Hann Weighting

Overlapping regions are blended using Hann window weights:
$$w(x) = \sin^2\left(\frac{\pi x}{L}\right)$$
where $L$ is the overlap width. This ensures smooth transitions and avoids edge artifacts.

### Resolution

- **Azimuth resolution**: Determined by sub-aperture length $M$ (not full aperture $N$)
- **Range resolution**: Same as spotlight PFA ($\rho_r = c / 2B$)
- **Strip length**: Extends to full collection track

### Modules Used

| Module | Purpose |
|--------|--------|
| `grdl.IO` | CPHD reader via factory (`get_reader('cphd', ...)`) |
| `grdl.image_processing.sar` | `StripmapPFA` |
| `napari` | Interactive visualization |

## Preamble — Version Check

In [ ]:
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(
        f"GRDL {required}+ required. Found {current}. "
        f"Update: pip install --upgrade grdl"
    )

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import napari

from grdl.IO import get_reader
from grdl.image_processing.sar.image_formation import StripmapPFA

## Configuration — User Paths

In [ ]:
# Path to CPHD file (stripmap collection)
CPHD_PATH = Path("/path/to/your/stripmap_cphd.cphd")

# Validate path
assert CPHD_PATH.exists(), f"CPHD file not found: {CPHD_PATH}"
assert CPHD_PATH.suffix.lower() in [".cphd", ".cph"], f"Expected CPHD file, got {CPHD_PATH.suffix}"

print(f"✓ CPHD file: {CPHD_PATH.name}")

## Step 1: Load CPHD Metadata and Signal

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    meta = reader.metadata
    
    # Extract collection parameters
    channel = meta.data.channels[0]
    n_vectors = channel.num_vectors
    n_samples = channel.num_samples
    
    print(f"Collection mode: {meta.data.radar_mode}")
    print(f"Phase history: {n_vectors} vectors × {n_samples} samples")
    print(f"Center frequency: {meta.data.tx_frequency_nominal / 1e9:.2f} GHz")
    
    # Read full signal
    signal = reader.read_signal(channel_id=channel.identifier)

print(f"\nSignal shape: {signal.shape}, dtype: {signal.dtype}")

## Step 2: Configure Stripmap PFA Parameters

In [ ]:
# Stripmap PFA configuration
stripmap_params = {
    'subaperture_pulses': 4096,    # Fixed sub-aperture length (or None for auto-sizing)
    'overlap': 0.5,                # 50% overlap between sub-apertures
    'grid_mode': 'inscribed',      # or 'circumscribed'
    'range_oversample': 1.0,
    'azimuth_oversample': 1.0,
    'projection': 'slant',
}

print("Stripmap PFA Configuration:")
for k, v in stripmap_params.items():
    print(f"  {k}: {v}")

## Step 3: Initialize Stripmap PFA Processor

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    stripmap_pfa = StripmapPFA(
        metadata=reader.metadata,
        **stripmap_params
    )
    
    # Query sub-aperture partition
    n_subapertures = len(stripmap_pfa.subaperture_indices)
    
    print(f"Stripmap PFA initialized")
    print(f"Sub-apertures: {n_subapertures}")
    print(f"Overlap fraction: {stripmap_params['overlap']}")

## Step 4: Form Strip Image

The `StripmapPFA.form()` method:
1. Partitions signal into overlapping blocks
2. Forms each block with PFA
3. Mosaics with Hann-weighted blending

In [ ]:
with get_reader('cphd', CPHD_PATH) as reader:
    stripmap_pfa = StripmapPFA(metadata=reader.metadata, **stripmap_params)
    signal = reader.read_signal()
    
    # Form the continuous strip image
    print("Running Stripmap PFA...")
    strip_image = stripmap_pfa.form(signal)

print(f"✓ Strip image formed: {strip_image.shape}, dtype: {strip_image.dtype}")

## Step 5: Compute Image Statistics

In [ ]:
magnitude = np.abs(strip_image)
phase = np.angle(strip_image)

print("Strip Image Statistics:")
print(f"  Magnitude — min: {magnitude.min():.2e}, max: {magnitude.max():.2e}")
print(f"  Magnitude — mean: {magnitude.mean():.2e}, std: {magnitude.std():.2e}")

# dB-scale magnitude
magnitude_db = 20 * np.log10(magnitude + 1e-12)
vmax = magnitude_db.max()
vmin = vmax - 50.0  # 50 dB dynamic range

print(f"  dB range: [{vmin:.1f}, {vmax:.1f}] dB")

## Step 6: Visualization — Interactive Napari Viewer

In [ ]:
viewer = napari.Viewer(title="Stripmap PFA")

# Layer 1: Magnitude (dB scale)
viewer.add_image(
    magnitude_db,
    name="Stripmap Magnitude (dB)",
    colormap="gray",
    contrast_limits=[vmin, vmax],
)

# Layer 2: Phase
viewer.add_image(
    phase,
    name="Phase (rad)",
    colormap="twilight",
    visible=False,
)

print("✓ Napari viewer launched")
print("  Strip extends along azimuth (vertical) axis")
print("  Zoom out to see full strip extent")

## Physical Interpretation

### Stripmap Geometry
- **Azimuth axis** (vertical): Along-track direction (flight path)
- **Range axis** (horizontal): Cross-track direction (slant range)
- **Continuous coverage**: Strip extends for entire collection duration

### Sub-aperture Effects
- **Smaller sub-apertures** → coarser azimuth resolution, faster processing
- **Larger sub-apertures** → finer azimuth resolution, slower processing
- **Overlap 0.5**: Standard choice balancing resolution and efficiency

### Mosaicking Artifacts
- **Seams**: Visible boundaries between sub-apertures (increase overlap if present)
- **Hann weighting**: Smooth blending in overlap regions
- **Edge effects**: First/last sub-apertures may have lower SNR

### Resolution vs Spotlight PFA
- **Stripmap**: Uniform (coarser) azimuth resolution along full strip
- **Spotlight**: Finest resolution at scene center, degrades at edges

## Summary

Successfully demonstrated:
- ✅ Stripmap CPHD loading via IO factory
- ✅ Sub-aperture partitioning with configurable overlap
- ✅ Automated mosaicking with Hann-weighted blending
- ✅ Continuous strip image formation
- ✅ Interactive napari visualization

### Key GRDL Patterns
- `get_reader('cphd', path)` for stripmap CPHD
- `StripmapPFA(metadata, subaperture_pulses=N, overlap=α)` for configuration
- Single `form(signal)` call handles full pipeline

### Next Steps
- Try different sub-aperture lengths (2048, 8192) — trade resolution vs processing time
- Adjust overlap (0.3, 0.7) — higher overlap = smoother seams, more computation
- Compare with RDA stripmap formation (next notebook)
- Export to SICD: `SICDWriter.write(strip_image, metadata, output_path)`